# VQPAINT tiny decoder (distilled VQGAN f16 decoder)

Trains small conv decoders that map `vqgan_imagenet_f16_16384` tokens to RGB, distilled from the original decoder, with on-the-fly token grids. Inputs: `vqpaint-assets` (checkpoint) and `vqpaint-tokens` (token grids). Output: `/kaggle/working/tiny/` (weights in the browser format, eval sheet, metrics).

In [ ]:
S = dict(
    HOURS=3.5,                      # training time budget (the session dies at 12 h)
    BATCH=16,
    VARIANTS="A:64,64,64,32,16:2,2,2,1,1;B:64,64,48,24,12:2,2,2,1,1",
    LPIPS=1.0, LR=2e-3,
    BRANCH="research/tiny-vqgan",
    RESUME_FROM="/kaggle/input/vqpaint-tiny-decoder*/tiny/ckpt.pt",   # a previous version's output added as input (optional)
)
import glob, os, shutil, subprocess, sys, time
TMP, WORK = "/kaggle/tmp", "/kaggle/working"; os.makedirs(TMP, exist_ok=True)
REPO, DATA, OUT = f"{TMP}/VQPAINT", f"{TMP}/data", f"{WORK}/tiny"
def sh(cmd):
    print("$", cmd, flush=True); subprocess.run(cmd, shell=True, check=True)
print(S)

In [ ]:
# setup: repo (research branch), taming-transformers, checkpoint from the assets dataset, token grids from the tokens dataset
if not os.path.isdir(REPO): sh(f"git clone -q --branch {S['BRANCH']} --depth 1 https://github.com/NazarGol/VQPAINT.git {REPO}")
TAM = f"{REPO}/web/export/taming-transformers"
if not os.path.isdir(TAM): sh(f"git clone -q --depth 1 https://github.com/CompVis/taming-transformers.git {TAM}")
ASSETS = next(iter(glob.glob("/kaggle/input/vqpaint-assets*")), None); assert ASSETS, "attach the vqpaint-assets dataset"
CK = f"{REPO}/web/export/checkpoints"
if not os.path.isdir(CK): os.symlink(f"{ASSETS}/checkpoints", CK)
TOK = next(iter(glob.glob("/kaggle/input/vqpaint-tokens*")), None); assert TOK, "attach the vqpaint-tokens dataset"
if not os.path.isdir(DATA): shutil.copytree(TOK, DATA)
sh(f"{sys.executable} -m pip install -q lpips onnxruntime")
if not os.path.exists(f"{DATA}/vision_model.onnx"):
    sh(f"curl -sSL https://huggingface.co/Xenova/mobileclip_s0/resolve/main/onnx/vision_model.onnx -o {DATA}/vision_model.onnx")
sh(f"ls -la {DATA} {CK}/ && nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
# resume from an earlier version's checkpoint when attached
os.makedirs(OUT, exist_ok=True)
prev = sorted(glob.glob(S["RESUME_FROM"])) if S["RESUME_FROM"] else []
if prev and not os.path.exists(f"{OUT}/ckpt.pt"): shutil.copy(prev[-1], f"{OUT}/ckpt.pt"); print("resuming from", prev[-1])

In [ ]:
# train (streams the log); checkpoints every 15 min into OUT so a killed session can resume
cmd = f"{sys.executable} -u {REPO}/web/research/tiny/train_decoder.py --data {DATA} --out {OUT} --hours {S['HOURS']} --batch {S['BATCH']} --variants '{S['VARIANTS']}' --lpips {S['LPIPS']} --lr {S['LR']} --clip {DATA}/vision_model.onnx"
print("$", cmd, flush=True)
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
with open(f"{OUT}/train.log", "a") as log:
    for line in p.stdout:
        if "Warning" in line or "warn(" in line: continue
        print(line, end="", flush=True); log.write(line)
print("exit", p.wait())

In [ ]:
from IPython.display import Image, display
import json
print(json.dumps(json.load(open(f"{OUT}/eval.json")), indent=1))
sh(f"ls -la {OUT}")
display(Image(f"{OUT}/eval_sheet.png", width=700))